# Valence vs. semantics in DINOv2: linear probes across model sizes

Runtime → Change runtime type → **T4 GPU**. Add a Colab secret `HF_TOKEN` (key icon) after accepting the SMID terms at https://huggingface.co/datasets/AIML-TUDA/smid.

SMID images live only on this runtime and are deleted in step 5. Features, results and figures are saved to Google Drive (`MyDrive/valence_probe`) so a disconnect loses nothing.

> Content warning: SMID contains disturbing images. Nothing here displays them except the step-8 gallery thumbnails, which you view by choice.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Scripts (copied from `src/`)

In [ ]:
%%writefile extract_features.py
"""
Extract per-layer DINOv2 features for (a) SMID images with human ratings and
(b) a reference semantic dataset (any ImageFolder: root/<class>/<img>).

Features per layer = concat(CLS token, mean of patch tokens), as in DINOv2's linear eval.
Saved as float16 arrays of shape [n_layers, n_images, 2*hidden_dim].

Usage:
  python extract_features.py --smid-dir data/smid --semantic-dir data/imagenette2-320/train \
      --models S B L G --out features
"""
import argparse, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel

MODELS = {
    "S": "facebook/dinov2-small",
    "B": "facebook/dinov2-base",
    "L": "facebook/dinov2-large",
    "G": "facebook/dinov2-giant",
}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RATINGS = ["valence", "arousal", "morality"]


# ----------------------------------------------------------------------------- SMID
def _find_rating_col(cols, key):
    """Pick the column for a rating: contains key, prefer 'mean', avoid sd/se/n columns."""
    keys = {"morality": ["moral"], "valence": ["valence"], "arousal": ["arousal"]}[key]
    cands = [c for c in cols if any(k in c.lower() for k in keys)]
    cands = [c for c in cands if not re.search(r"(^|_)(sd|se|std|n|var)($|_)", c.lower())]
    if not cands:
        return None
    mean = [c for c in cands if "mean" in c.lower()]
    return (mean or cands)[0]


def load_smid(root, overrides):
    root = Path(root)
    images = {p.stem: p for p in root.rglob("*") if p.suffix.lower() in IMG_EXT}
    if not images:
        raise SystemExit(f"No images found under {root}")

    csvs = sorted(root.rglob("*.csv"))
    meta = None
    for c in csvs:  # prefer a norms/metadata file, else concat train+test
        if any(k in c.name.lower() for k in ["norm", "metadata"]):
            meta = pd.read_csv(c)
            print(f"[smid] metadata: {c}")
            break
    if meta is None:
        parts = [pd.read_csv(c) for c in csvs if c.stem.lower() in ("train", "test")]
        if not parts:
            raise SystemExit(f"No metadata csv found under {root}. Found: {[c.name for c in csvs]}")
        meta = pd.concat(parts, ignore_index=True)
        print(f"[smid] metadata: concatenated {len(parts)} split files")

    # image-id column = the column whose values best match image file stems
    def match_rate(col):
        vals = meta[col].astype(str).map(lambda s: Path(s).stem)
        return vals.isin(images.keys()).mean()

    id_col = overrides.get("id") or max(meta.columns, key=match_rate)
    if match_rate(id_col) < 0.5:
        raise SystemExit(f"Could not match metadata rows to image files. Columns: {list(meta.columns)}; "
                         f"sample image stems: {list(images)[:5]}. Use --col-id.")

    cols = {}
    for r in RATINGS:
        cols[r] = overrides.get(r) or _find_rating_col(meta.columns, r)
        if cols[r] is None:
            raise SystemExit(f"No column for '{r}'. Columns: {list(meta.columns)}. Use --col-{r}.")
    print(f"[smid] id column: {id_col}; rating columns: {cols}")

    meta = meta.assign(_stem=meta[id_col].astype(str).map(lambda s: Path(s).stem))
    meta = meta[meta._stem.isin(images.keys())].drop_duplicates("_stem").reset_index(drop=True)
    labels = pd.DataFrame({"image": meta._stem, **{r: meta[cols[r]].astype(float) for r in RATINGS}})
    labels = labels.dropna().reset_index(drop=True)
    paths = [images[s] for s in labels.image]
    print(f"[smid] {len(paths)} images with all ratings")
    return paths, labels


# ------------------------------------------------------------------------- semantic
def load_imagefolder(root, max_per_class, seed=0):
    root = Path(root)
    rng = np.random.default_rng(seed)
    paths, labels, classes = [], [], sorted(d.name for d in root.iterdir() if d.is_dir())
    for ci, c in enumerate(classes):
        files = sorted(p for p in (root / c).rglob("*") if p.suffix.lower() in IMG_EXT)
        if len(files) > max_per_class:
            files = [files[i] for i in sorted(rng.choice(len(files), max_per_class, replace=False))]
        paths += files
        labels += [ci] * len(files)
    print(f"[semantic] {len(paths)} images, {len(classes)} classes")
    return paths, np.array(labels), classes


# ------------------------------------------------------------------------ features
@torch.no_grad()
def extract(model_name, paths, batch_size, device):
    proc = AutoImageProcessor.from_pretrained(model_name)  # resize 256 + center-crop 224
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModel.from_pretrained(model_name).to(device=device, dtype=dtype).eval()
    n_reg = getattr(model.config, "num_register_tokens", 0) or 0
    feats = None
    for i in tqdm(range(0, len(paths), batch_size), desc=model_name):
        ims = [Image.open(p).convert("RGB") for p in paths[i:i + batch_size]]
        x = proc(images=ims, return_tensors="pt")["pixel_values"].to(device, dtype)
        hs = model(pixel_values=x, output_hidden_states=True).hidden_states[1:]  # skip embeddings
        layer_feats = []
        for h in hs:
            cls, patches = h[:, 0], h[:, 1 + n_reg:]
            layer_feats.append(torch.cat([cls, patches.mean(1)], dim=-1).float().cpu().numpy())
        batch = np.stack(layer_feats).astype(np.float16)  # [L, B, 2D]
        if feats is None:
            feats = np.zeros((batch.shape[0], len(paths), batch.shape[2]), np.float16)
        feats[:, i:i + len(ims)] = batch
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return feats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--smid-dir", required=True)
    ap.add_argument("--semantic-dir", required=True, help="ImageFolder root: <root>/<class>/<image>")
    ap.add_argument("--semantic-max-per-class", type=int, default=300)
    ap.add_argument("--models", nargs="+", default=list(MODELS), choices=list(MODELS))
    ap.add_argument("--out", default="features")
    ap.add_argument("--batch-size", type=int, default=32)
    for r in ["id"] + RATINGS:
        ap.add_argument(f"--col-{r}", default=None, help=f"override SMID column for {r}")
    a = ap.parse_args()

    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    overrides = {r: getattr(a, f"col_{r}") for r in ["id"] + RATINGS if getattr(a, f"col_{r}")}

    smid_paths, smid_labels = load_smid(a.smid_dir, overrides)
    smid_labels.to_csv(out / "smid_labels.csv", index=False)
    sem_paths, sem_labels, classes = load_imagefolder(a.semantic_dir, a.semantic_max_per_class)
    np.save(out / "semantic_labels.npy", sem_labels)
    json.dump(classes, open(out / "semantic_classes.json", "w"))

    for m in a.models:
        for tag, paths in [("smid", smid_paths), ("semantic", sem_paths)]:
            f = out / f"{m}_{tag}.npy"
            if f.exists():
                print(f"skip {f} (exists)"); continue
            np.save(f, extract(MODELS[m], paths, a.batch_size, device))
            print(f"saved {f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_probes.py
"""
Linear probes on cached DINOv2 features: how decodable is affective valence
(plus arousal, morality) compared with ordinary semantic categories, per layer and per model size?

Tasks
  Binary (all subsampled to the same N, balanced, metric = AUROC):
    smid:valence / arousal / morality  -> top third vs bottom third of the human rating
    semantic:<class>                   -> one class vs the rest
  Continuous (full SMID, metric = Spearman rho and R^2 of out-of-fold predictions):
    valence / arousal / morality

Probe per fold: StandardScaler -> PCA(k, fit on train) -> RidgeCV (alpha by efficient LOO).

Usage:
  python run_probes.py --features features --out results
  python run_probes.py --synthetic --out results_synthetic     # pipeline smoke test, no data needed
"""
import argparse, json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PARAMS = {"S": 22e6, "B": 86e6, "L": 304e6, "G": 1.1e9}
RATINGS = ["valence", "arousal", "morality"]
ALPHAS = np.logspace(-1, 5, 13)


def probe(k):
    return make_pipeline(StandardScaler(), PCA(k, random_state=0), RidgeCV(alphas=ALPHAS))


def oof_predict(X, y, folds, k, stratify, seed):
    k = min(k, X.shape[1], int(len(y) * (folds - 1) / folds) - 1)
    cv = (StratifiedKFold if stratify else KFold)(folds, shuffle=True, random_state=seed)
    pred = np.zeros(len(y))
    for tr, te in cv.split(X, y if stratify else None):
        pred[te] = probe(k).fit(X[tr], y[tr]).predict(X[te])
    return pred


def boot_auc(y, s, n, rng):
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            vals.append(roc_auc_score(y[i], s[i]))
    return np.percentile(vals, [2.5, 97.5])


def build_binary_tasks(smid, sem_y, classes, n_binary, rng):
    """Return {task_name: (dataset, indices, binary_labels)}, every task balanced with n_binary samples."""
    tasks = {}
    half = n_binary // 2
    for r in RATINGS:
        v = smid[r].to_numpy()
        lo, hi = np.where(v <= np.quantile(v, 1 / 3))[0], np.where(v >= np.quantile(v, 2 / 3))[0]
        h = min(half, len(lo), len(hi))
        idx = np.r_[rng.choice(lo, h, replace=False), rng.choice(hi, h, replace=False)]
        tasks[f"smid:{r}"] = ("smid", idx, np.r_[np.zeros(h), np.ones(h)])
    for ci, c in enumerate(classes):
        pos, neg = np.where(sem_y == ci)[0], np.where(sem_y != ci)[0]
        h = min(half, len(pos), len(neg))
        idx = np.r_[rng.choice(neg, h, replace=False), rng.choice(pos, h, replace=False)]
        tasks[f"semantic:{c}"] = ("semantic", idx, np.r_[np.zeros(h), np.ones(h)])
    sizes = {t: len(v[1]) for t, v in tasks.items()}
    if len(set(sizes.values())) > 1:
        print(f"[warn] unequal binary task sizes (lower --n-binary to equalise): {sizes}")
    return tasks


def run(feats, smid, sem_y, classes, models, a):
    rng = np.random.default_rng(a.seed)
    tasks = build_binary_tasks(smid, sem_y, classes, a.n_binary, rng)
    rows = []
    for m in models:
        F = {"smid": feats[m]["smid"], "semantic": feats[m]["semantic"]}
        n_layers = F["smid"].shape[0]
        layers = sorted(set(range(0, n_layers, a.layer_stride)) | {n_layers - 1})
        for L in layers:
            depth = (L + 1) / n_layers
            for t, (ds, idx, yb) in tasks.items():
                X = F[ds][L, idx].astype(np.float32)
                s = oof_predict(X, yb, a.folds, a.pca, True, a.seed)
                lo, hi = boot_auc(yb, s, a.boot, rng)
                rows.append(dict(model=m, params=PARAMS[m], layer=L, depth=depth, task=t,
                                 kind="binary", metric="auroc", value=roc_auc_score(yb, s), lo=lo, hi=hi))
            X = F["smid"][L].astype(np.float32)
            for r in RATINGS:
                y = smid[r].to_numpy()
                p = oof_predict(X, y, a.folds, a.pca, False, a.seed)
                rows.append(dict(model=m, params=PARAMS[m], layer=L, depth=depth, task=f"smid:{r}",
                                 kind="continuous", metric="spearman", value=spearmanr(y, p)[0]))
                rows.append(dict(model=m, params=PARAMS[m], layer=L, depth=depth, task=f"smid:{r}",
                                 kind="continuous", metric="r2", value=r2_score(y, p)))
            print(f"{m} layer {L + 1}/{n_layers} done")
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------- plots
COLORS = {"S": "#9ecae1", "B": "#4292c6", "L": "#2171b5", "G": "#08306b"}
TASK_COLORS = {"smid:valence": "#d94801", "smid:arousal": "#fd8d3c",
               "smid:morality": "#7a0177", "semantic (mean)": "#238b45"}


def add_semantic_mean(df):
    sem = df[df.task.str.startswith("semantic:")]
    agg = sem.groupby(["model", "params", "layer", "depth", "kind", "metric"], as_index=False)[
        ["value", "lo", "hi"]].mean()
    return pd.concat([df, agg.assign(task="semantic (mean)")], ignore_index=True)


def plot_layers(df, out):
    d = df[(df.metric == "auroc") & df.task.isin(TASK_COLORS)]
    tasks = list(TASK_COLORS)
    fig, axes = plt.subplots(1, len(tasks), figsize=(4 * len(tasks), 3.4), sharey=True)
    for ax, t in zip(axes, tasks):
        for m in d.model.unique():
            s = d[(d.task == t) & (d.model == m)].sort_values("depth")
            ax.plot(s.depth, s.value, "-o", ms=3, color=COLORS.get(m), label=f"DINOv2-{m}")
        ax.set_title(t); ax.set_xlabel("relative depth"); ax.grid(alpha=.3)
    axes[0].set_ylabel("AUROC (out-of-fold)"); axes[-1].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out / "fig1_auroc_by_layer.png", dpi=150); plt.close(fig)


def best_layer(df, metric):
    d = df[df.metric == metric]
    return d.loc[d.groupby(["model", "task"]).value.idxmax()]


def plot_scaling(df, out):
    b = best_layer(df, "auroc")
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for t, c in TASK_COLORS.items():
        s = b[b.task == t].sort_values("params")
        if s.empty:
            continue
        ax.errorbar(s.params, s.value, yerr=[s.value - s.lo, s.hi - s.value], fmt="-o", color=c,
                    capsize=3, label=t)
    ax.set_xscale("log"); ax.set_xlabel("parameters"); ax.set_ylabel("best-layer AUROC")
    ax.set_title("Decodability vs DINOv2 size"); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out / "fig2_scaling_auroc.png", dpi=150); plt.close(fig)

    b = best_layer(df, "spearman")
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for t in [f"smid:{r}" for r in RATINGS]:
        s = b[b.task == t].sort_values("params")
        ax.plot(s.params, s.value, "-o", color=TASK_COLORS[t], label=t)
    ax.set_xscale("log"); ax.set_xlabel("parameters"); ax.set_ylabel("best-layer Spearman ρ")
    ax.set_title("Continuous ratings (full SMID)"); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out / "fig3_scaling_continuous.png", dpi=150); plt.close(fig)


# ------------------------------------------------------------------------- synthetic
def synthetic(models, n_smid=900, n_sem=1000, n_cls=5, seed=0):
    """Fake features with planted signals: semantic easy & early, valence weaker & late."""
    rng = np.random.default_rng(seed)
    dims = {"S": 384, "B": 768, "L": 1024, "G": 1536}
    nl = {"S": 12, "B": 12, "L": 24, "G": 40}
    z = rng.normal(size=(n_smid, 3))  # latent valence, arousal, morality
    z[:, 2] = 0.7 * z[:, 0] + 0.7 * z[:, 2]
    smid = pd.DataFrame({"image": [f"img{i}" for i in range(n_smid)],
                         **{r: 3 + z[:, i] + 0.5 * rng.normal(size=n_smid) for i, r in enumerate(RATINGS)}})
    sem_y = rng.integers(0, n_cls, n_sem)
    feats = {}
    for m in models:
        D, L = 2 * dims[m], nl[m]
        size_gain = np.log10(PARAMS[m]) - 7
        W = rng.normal(size=(3, D)); C = rng.normal(size=(n_cls, D))
        fs, fm = [], []
        for l in range(L):
            d = (l + 1) / L
            fs.append(rng.normal(size=(n_smid, D)) + (0.04 * size_gain * d ** 2) * z @ W)
            fm.append(rng.normal(size=(n_sem, D)) + (0.15 * min(1, 2 * d)) * C[sem_y])
        feats[m] = {"smid": np.stack(fs).astype(np.float16), "semantic": np.stack(fm).astype(np.float16)}
    return feats, smid, sem_y, [f"class{i}" for i in range(n_cls)]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="features")
    ap.add_argument("--out", default="results")
    ap.add_argument("--models", nargs="+", default=["S", "B", "L", "G"])
    ap.add_argument("--n-binary", type=int, default=600, help="samples per binary task (balanced)")
    ap.add_argument("--pca", type=int, default=256, help="PCA dims (same for all sizes; also speeds up)")
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--layer-stride", type=int, default=1)
    ap.add_argument("--boot", type=int, default=200)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--synthetic", action="store_true")
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    if a.synthetic:
        feats, smid, sem_y, classes = synthetic(a.models)
    else:
        fd = Path(a.features)
        smid = pd.read_csv(fd / "smid_labels.csv")
        sem_y = np.load(fd / "semantic_labels.npy")
        classes = json.load(open(fd / "semantic_classes.json"))
        models = [m for m in a.models if (fd / f"{m}_smid.npy").exists()]
        feats = {m: {t: np.load(fd / f"{m}_{t}.npy", mmap_mode="r") for t in ("smid", "semantic")}
                 for m in models}
        a.models = models

    df = add_semantic_mean(run(feats, smid, sem_y, classes, a.models, a))
    df.to_csv(out / "results.csv", index=False)
    plot_layers(df, out); plot_scaling(df, out)

    summary = best_layer(df, "auroc")
    summary = summary[summary.task.isin(TASK_COLORS)].pivot(index="task", columns="model", values="value")
    print("\nBest-layer AUROC:\n", summary[[m for m in a.models if m in summary.columns]].round(3))
    print(f"\nWrote {out}/results.csv and figures.")


if __name__ == "__main__":
    main()


## 2. Imagenette (semantic reference)

In [ ]:
!mkdir -p data && wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz && tar xzf imagenette2-320.tgz -C data && rm imagenette2-320.tgz && echo "imagenette ok"

## 3. SMID (gated)

In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download
snapshot_download("AIML-TUDA/smid", repo_type="dataset", local_dir="data/smid", token=userdata.get("HF_TOKEN"))

## 4. Per-layer features for DINOv2 S/B/L/G (~15–25 min on a T4)
Check the printed SMID columns (`file_name`, `valence_mean`, `arousal_mean`, `moral_mean`).

In [ ]:
!python extract_features.py --smid-dir data/smid --semantic-dir data/imagenette2-320/train --models S B L G --out features

## 5. Delete the SMID images

In [ ]:
!rm -rf data/smid && echo "SMID images deleted"

## 6. Save to Drive, then probes one model size at a time
Every second layer plus the last. Each size's results land on Drive as soon as it finishes (~5 / 10 / 20 / 35 min on Colab's CPU).

In [ ]:
import os, shutil, subprocess, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/valence_probe'
os.makedirs(DRIVE, exist_ok=True)
for f in ['features', 'extract_features.py', 'run_probes.py']:
    if not os.path.exists(f'{DRIVE}/{f}'):
        (shutil.copytree if os.path.isdir(f) else shutil.copy)(f, f'{DRIVE}/{f}')

def show(m):
    df = pd.read_csv(f'{DRIVE}/results_{m}/results.csv')
    t = df[(df.metric == 'auroc') & df.task.isin(['smid:valence', 'smid:arousal', 'smid:morality', 'semantic (mean)'])]
    print(f'--- DINOv2-{m}: AUROC by layer ---')
    print(t.pivot_table(index='layer', columns='task', values='value').round(3).to_string())
    c = df[df.metric == 'spearman'].pivot_table(index='layer', columns='task', values='value')
    print('best-layer Spearman rho:', c.max().round(3).to_dict(), '\n', flush=True)

for m in ['S', 'B', 'L', 'G']:
    if not os.path.exists(f'{DRIVE}/results_{m}/results.csv'):
        print(f'=== running DINOv2-{m} ===', flush=True)
        r = subprocess.run(['python', 'run_probes.py', '--features', 'features', '--out', f'{DRIVE}/results_{m}',
                            '--models', m, '--layer-stride', '2'], capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-3000:]); break
    show(m)

## 7. Combine all sizes and plot

In [ ]:
import pandas as pd, run_probes as rp
from pathlib import Path
from IPython.display import Image, display
D = Path('/content/drive/MyDrive/valence_probe')
df = pd.concat([pd.read_csv(D / f"results_{m}/results.csv") for m in "SBLG" if (D / f"results_{m}/results.csv").exists()], ignore_index=True)
out = D / "results"; out.mkdir(exist_ok=True); df.to_csv(out / "results.csv", index=False)
rp.plot_layers(df, out); rp.plot_scaling(df, out)
for f in ["fig1_auroc_by_layer", "fig2_scaling_auroc", "fig3_scaling_continuous"]:
    display(Image(str(out / f"{f}.png")))

## 8. Gallery: 5 images per SMID rating scale (all 8 scales)
Downloads only the 40 chosen images, saves 224-px thumbnails to Drive (`gallery_thumbs.json`), deletes the originals. Use with `src/make_figures.py --gallery ... --thumbs ...`.

In [ ]:
import os, io, json, base64, shutil, pandas as pd
from PIL import Image
from google.colab import drive, userdata
from huggingface_hub import hf_hub_download, snapshot_download
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/valence_probe'
tok = userdata.get('HF_TOKEN')
meta = pd.read_csv(hf_hub_download('AIML-TUDA/smid', 'metadata.csv', repo_type='dataset', token=tok))
SCALES = ['valence', 'arousal', 'moral', 'harm', 'fairness', 'ingroup', 'authority', 'purity']
Q = [0.03, 0.25, 0.5, 0.75, 0.97]
rows, chosen = [], set()
for s in SCALES:
    col, picked = f'{s}_mean', []
    for q in Q:
        cand = meta.loc[~meta.file_name.isin(picked)].copy()
        cand['d'] = (cand[col] - meta[col].quantile(q)).abs()
        picked.append(cand.sort_values('d').file_name.iloc[0])
    rows.append((s, picked)); chosen.update(picked)
local = snapshot_download('AIML-TUDA/smid', repo_type='dataset', local_dir='gallery_tmp', token=tok,
                          allow_patterns=[p for fn in chosen for p in (fn, '*/' + fn)])
paths = {f: os.path.join(r, f) for r, _, fs in os.walk(local) for f in fs if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
def thumb(fn, side=224):
    im = Image.open(paths[fn]).convert('RGB'); im.thumbnail((side, side))
    b = io.BytesIO(); im.save(b, 'JPEG', quality=78); return base64.b64encode(b.getvalue()).decode()
cols = [f'{s}_mean' for s in SCALES]
gallery = {'quantiles': Q, 'rows': [{'scale': s, 'items': [{'file': fn, 'img': thumb(fn), **{c[:-5]: round(float(meta.loc[meta.file_name == fn, c].iloc[0]), 2) for c in cols}} for fn in picked]} for s, picked in rows]}
json.dump(gallery, open(f'{DRIVE}/gallery_thumbs.json', 'w'))
shutil.rmtree('gallery_tmp', ignore_errors=True)
print('saved', sum(len(r['items']) for r in gallery['rows']), 'thumbnails to', f'{DRIVE}/gallery_thumbs.json')